# YT Agent — Kaggle GPU worker

Triggered by the `kaggle-dispatch.yml` GitHub Actions workflow whenever the
Vercel `/api/maintenance/needs-worker` route reports that a render is queued
and no GPU worker is alive. Boots an ephemeral FastAPI backend on a free
Kaggle T4 / P100, registers itself in Firestore, claims the queued job, runs
the pipeline, then self-terminates after 10 min of idle to preserve the
30 GPU hr/week budget.

**One-time setup**: see `kaggle/README.md`. You need to add
`GOOGLE_APPLICATION_CREDENTIALS_JSON` (and optionally R2 / SFTP) to the
Kaggle Add-ons → Secrets panel for THIS notebook.

In [ ]:
# 1) System deps + repo clone
import os, subprocess
# DEBIAN_FRONTEND=noninteractive so any apt post-install prompt
# doesn't stall the whole boot silently (fontconfig sometimes prompts
# on a fresh Kaggle image).
os.environ['DEBIAN_FRONTEND'] = 'noninteractive'
subprocess.check_call(['apt-get', '-qq', 'update'])
# fonts-noto{,-cjk,-color-emoji} + fonts-dejavu-core — needed for
# burned-in captions to render non-Latin scripts (Devanagari, Bengali,
# CJK, Arabic, Thai). DejaVu Sans is the primary font (bundled default)
# with libass falling back to Noto CJK for CJK. -y and DEBIAN_FRONTEND
# ensure no interactive prompt.
subprocess.check_call(['apt-get', '-qq', 'install', '-y',
                       '--no-install-recommends',
                       'ffmpeg', 'fonts-dejavu-core',
                       'fonts-noto', 'fonts-noto-cjk',
                       'fonts-noto-color-emoji'])
subprocess.run(['fc-cache', '-f'], check=False)

REPO_URL = 'https://github.com/Ahsan3301/yt_agent.git'
BRANCH   = 'main'
REPO_DIR = '/kaggle/working/yt_agent'
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
os.chdir(REPO_DIR)
print('repo at:', os.getcwd())
print('commit:', subprocess.check_output(['git', 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# 2) Python deps — batched where safe, split when a per-package flag
#    would otherwise leak to every package in the same pip call.
#    Was 10 pip subprocess calls; now 5. Chromium install runs in the
#    background so cell exits before it finishes.
import subprocess, sys, os

# hf_transfer env var MUST be set before any huggingface_hub import.
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# apt-installed system deps (fast).
subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'espeak-ng'])

# Kaggle's base image ships vanilla `phonemizer` which misaki (Kokoro's
# phoneme dep) can't use — its EspeakWrapper doesn't have
# set_data_path(). Uninstall FIRST so the phonemizer-fork install
# below actually wins on `import phonemizer`.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'phonemizer'], check=False)

# MAIN BATCH — no per-package flags. Any --force-reinstall / --upgrade
# in a pip call applies to EVERY package listed, not just the one
# next to it. Confirmed live 2026-07-09: leaving --force-reinstall
# in the batch reinstalled torch/torchvision from mismatched PyPI
# wheels, broke torchvision.extension, and killed transformers +
# Kokoro + diffusers on import. Do NOT re-merge the flagged installs
# back into this batch.
#
# transformers CAPPED at <4.50 — anything newer removes AlbertModel
# (Kokoro's backbone) + PreTrainedModel lazy-import (diffusers).
# pyOpenSSL >= 25 + cryptography >= 45 pinned to keep both on the
# NEW crypto C-binding API (avoids the GEN_EMAIL crash).
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-input',
    '-r', 'requirements.txt',
    '-r', 'requirements-browser.txt',
    'decord==0.6.0', 'av==12.3.0',
    'hf_transfer',
    'espeakng-loader>=0.2',
    'soundfile>=0.12',
    'kokoro>=0.9.4', 'misaki>=0.9.4',
    'transformers>=4.44,<4.50', 'accelerate>=0.30', 'safetensors>=0.4',
    'huggingface_hub>=0.23', 'peft>=0.10',
    'pyOpenSSL>=25.0.0', 'cryptography>=45.0.0',
])

# SEPARATE — phonemizer-fork needs --force-reinstall so pip doesn't
# skip if a stale wheel is cached. Kept OUT of the main batch above
# so the flag doesn't force-reinstall torch/torchvision from PyPI
# (which don't match Kaggle's preinstalled CUDA wheels).
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-input',
    '--force-reinstall', 'phonemizer-fork>=3.3.2',
])

# SEPARATE — edge-tts >=7 with --upgrade to dodge Microsoft auth-
# endpoint breakage in older cached wheels. Same reason as above:
# --upgrade would force torch/torchvision to move too if left in
# the main batch.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-input',
    '--upgrade', 'edge-tts>=7.0.0',
])

# SEPARATE — diffusers with --no-deps. Protects Kaggle's preinstalled
# CUDA torch from being clobbered by whatever torch diffusers'
# dependency spec would drag in.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'diffusers>=0.30',
])

# Playwright chromium install (~30-60s of Chromium download + apt libs)
# kicked in the BACKGROUND so cell 2 exits without waiting. The
# research_agent's is_available() gate falls back to keyword-only
# research if chromium isn't ready when the first job arrives.
_playwright_log = open('/tmp/playwright-install.log', 'w')
_playwright_bg = subprocess.Popen(
    [sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'],
    stdout=_playwright_log, stderr=subprocess.STDOUT,
)
print(f'playwright chromium install running in background (pid={_playwright_bg.pid}) - log at /tmp/playwright-install.log', flush=True)

# Import sanity checks so the log tells us what's ready before cell 5.
import torch
print('torch:', torch.__version__, 'cuda available:', torch.cuda.is_available())
try:
    import torchvision
    print('torchvision:', torchvision.__version__)
except Exception as e:
    print('torchvision: NOT ready ->', e)
try:
    import transformers
    print('transformers:', transformers.__version__)
except Exception as e:
    print('transformers: NOT ready ->', e)
try:
    from kokoro import KPipeline
    print('kokoro: ready')
except Exception as e:
    print('kokoro: NOT ready ->', e)
try:
    import edge_tts
    print('edge-tts:', getattr(edge_tts, '__version__', 'installed'))
except Exception as e:
    print('edge-tts: NOT ready ->', e)
try:
    from diffusers import AutoPipelineForText2Image  # noqa: F401
    print('diffusers: ready (local_sdxl provider available)')
except Exception as e:
    print('diffusers: NOT ready ->', e)
try:
    import OpenSSL, cryptography
    print(f'crypto: pyOpenSSL={OpenSSL.__version__} cryptography={cryptography.__version__}')
except Exception as e:
    print('crypto: NOT ready ->', e)

In [ ]:
# 3) BOOTSTRAP secrets.
#
# Two sources, in priority order:
#
#   (1) A private Kaggle Dataset attached via kernel-metadata.json. This
#       is the production path — survives `kaggle kernels push` cycles
#       (each push creates a new kernel version, but the dataset stays
#       attached). Create one private Dataset named e.g.
#       "yt-agent-secrets" with ONE file:
#
#           secrets.env
#               COOLIFY_BASE_URL=https://yt-agent.thyker.online
#               PB_URL=https://yt-agent.thyker.online/pb
#               POCKETBASE_ADMIN_EMAIL=admin@yt-agent.thyker.online
#               POCKETBASE_ADMIN_PASSWORD=your-password
#               RENDER_TRIGGER_KEY=...
#               STORAGE_PROVIDERS_ENC_KEY=...
#
#       Then reference it in kernel-metadata.json:
#           "dataset_sources": ["<your-username>/yt-agent-secrets"]
#
#   (2) Kaggle's Secrets panel (Add-ons → Secrets). Useful for
#       interactive testing. Same key names as the .env file.
#
# Either source works — the notebook reads whichever it finds first.
import os, glob

REQUIRED = [
    'COOLIFY_BASE_URL',
    'PB_URL',
    'POCKETBASE_ADMIN_EMAIL',
    'POCKETBASE_ADMIN_PASSWORD',
    'RENDER_TRIGGER_KEY',
    'STORAGE_PROVIDERS_ENC_KEY',
]
OPTIONAL = [
    # Legacy Vercel + Firestore path — keep working for users who
    # haven\'t migrated to Coolify yet.
    'GOOGLE_APPLICATION_CREDENTIALS_JSON',
    'GOOGLE_APPLICATION_CREDENTIALS_JSON_B64',
    'INSTANCE_LABEL', 'INSTANCE_TIER',
]
ALL_KEYS = REQUIRED + OPTIONAL


def _load_from_dataset_envfile():
    """Look for an attached Dataset secrets file. Kaggle mounts datasets
    under /kaggle/input/<dataset-name>/. Search for the most common
    file names."""
    candidates = []
    for root in glob.glob('/kaggle/input/*/'):
        for name in ('secrets.env', '.env', 'env.txt', 'secrets.txt'):
            p = os.path.join(root, name)
            if os.path.exists(p):
                candidates.append(p)
    if not candidates:
        return False
    found = False
    for path in candidates:
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#') or '=' not in line:
                    continue
                k, v = line.split('=', 1)
                k = k.strip()
                v = v.strip().strip('"').strip("'")
                if k in ALL_KEYS and v:
                    os.environ.setdefault(k, v)
                    found = True
        print(f'  loaded secrets from {path}')
    return found


def _load_from_kaggle_secrets():
    """Read from the Kaggle Secrets panel. Fails silently when not
    available (e.g. running in a Colab clone of the notebook)."""
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        loaded = 0
        for k in ALL_KEYS:
            try:
                v = secrets.get_secret(k)
            except Exception:
                continue
            if v and not os.environ.get(k):
                os.environ[k] = v
                loaded += 1
        if loaded:
            print(f'  loaded {loaded} secrets from Kaggle Secrets panel')
        return loaded > 0
    except Exception as e:
        print(f'  Kaggle Secrets unavailable ({e!r}); using Dataset only')
        return False


# Try both sources — Dataset first, then Kaggle Secrets fills any gaps.
got_dataset = _load_from_dataset_envfile()
got_secrets = _load_from_kaggle_secrets()
if not (got_dataset or got_secrets):
    raise SystemExit(
        'No secrets source found. Either attach the yt-agent-secrets '
        'Dataset (via kernel-metadata.json dataset_sources) OR add the '
        'required keys to the Kaggle Secrets panel.'
    )

missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    raise SystemExit('MISSING REQUIRED SECRETS: ' + ', '.join(missing))

# Defaults for the new outbound-poll mode.
os.environ.setdefault('DB_BACKEND', 'pocketbase')
os.environ.setdefault('STORAGE_BACKEND', 'registry')
os.environ.setdefault('WORKER_MODE', 'outbound_poll')
os.environ.setdefault('INSTANCE_TIER', 'gpu')
os.environ.setdefault('INSTANCE_LABEL', 'kaggle-gpu')
# Idle auto-shutdown ~25 min — preserves the 30 GPU-hr/week budget.
os.environ.setdefault('KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS', '1500')

print('Kaggle bootstrap OK')
print('  Dashboard:', os.environ.get('COOLIFY_BASE_URL'))
print('  Pocketbase:', os.environ.get('PB_URL'))
print('  Mode:', os.environ.get('WORKER_MODE'), '| tier:', os.environ.get('INSTANCE_TIER'))


In [ ]:
# 4) (Skipped in outbound-poll mode.) Legacy cloudflared tunnel cell.
import os, subprocess, time, re

if os.environ.get('WORKER_MODE', 'tunnel').lower() != 'tunnel':
    print('outbound-poll mode — no cloudflared tunnel needed.')
else:
    if not os.path.exists('/usr/local/bin/cloudflared'):
        subprocess.check_call([
            'wget', '-q',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-O', '/usr/local/bin/cloudflared',
        ])
        subprocess.check_call(['chmod', '+x', '/usr/local/bin/cloudflared'])
    tunnel_log = '/tmp/cloudflared.log'
    subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000',
         '--logfile', tunnel_log, '--loglevel', 'info'],
        stdout=open(tunnel_log + '.stdout', 'w'),
        stderr=subprocess.STDOUT,
    )
    url = None
    for _ in range(40):
        time.sleep(0.5)
        if not os.path.exists(tunnel_log):
            continue
        with open(tunnel_log) as f:
            m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', f.read())
            if m:
                url = m.group(0)
                break
    if not url:
        raise RuntimeError('cloudflared did not produce a URL')
    os.environ['PUBLIC_BACKEND_URL'] = url
    print('Public backend URL:', url)


In [ ]:
# 4.5) Pre-warm SDXL weights in the BACKGROUND — doesn't block boot.
#
# The heavy lift is downloading the ~7 GB SDXL safetensors into the
# HF cache. Previously this cell blocked cell 5 (uvicorn boot) for
# 60-120s so the dashboard's Monitor page couldn't see the worker
# until the download finished. Now snapshot_download runs in a
# daemon thread so uvicorn boots + the worker heartbeats immediately.
#
# When the first render calls _local_sdxl_load(), if the download is
# still running it briefly blocks there instead of at boot. hf_transfer
# usually finishes in <60s anyway, so the first render doesn't wait.
import os, time, subprocess, threading
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '180')

def _bg_snapshot_sdxl():
    try:
        t0 = time.time()
        print('preboot(bg): snapshotting SDXL weights into HF cache...', flush=True)
        from huggingface_hub import snapshot_download
        _sdxl_model = os.environ.get('LOCAL_SDXL_MODEL', 'stabilityai/sdxl-turbo')
        snapshot_download(
            _sdxl_model,
            allow_patterns=['*.json', '*.txt', '*.safetensors'],
        )
        print(f'preboot(bg): SDXL weights cached in {time.time()-t0:.1f}s', flush=True)
    except Exception as e:
        print(f'preboot(bg): SDXL cache warm skipped ({e!r})', flush=True)

threading.Thread(target=_bg_snapshot_sdxl, name='sdxl-snapshot', daemon=True).start()
print('preboot: SDXL snapshot kicked in background thread', flush=True)

# nvidia-smi visibility check — cheap and useful.
try:
    out = subprocess.check_output(['nvidia-smi', '-L'], timeout=5).decode().strip()
    print('preboot: nvidia-smi -L:\n' + out, flush=True)
    if out.count('GPU ') < 2:
        print('preboot: WARNING - only 1 GPU visible. To use T4x2, set '
              'Notebook Settings -> Accelerator -> GPU T4 x2 -> Save Version.',
              flush=True)
except Exception as e:
    print(f'preboot: nvidia-smi -L failed ({e!r})', flush=True)

print('preboot: done (main thread not blocked on SDXL)', flush=True)

In [ ]:
# 4.6) Self-check — verify multi-GPU + multilingual wiring
#
# Wrapped in try/except so any check crash still lets uvicorn boot.
# The self-check is diagnostic, not a gate.
try:
    from backend import self_check
    print(self_check.run(text=True), flush=True)
except Exception as e:
    print(f'self_check skipped ({e!r})', flush=True)

In [ ]:
# 5) Boot the backend (BLOCKING). idle_watchdog will os._exit(0) after
#    KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS once the queue is empty.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'uvicorn', 'backend.server:app',
    '--host', '0.0.0.0', '--port', '8000',
    '--log-level', 'info',
])